<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.1-burgers-2d/Ex09.1_01_burgers2d_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.1 · Notebook 01 — write the residual and the loss

**Paired with L9.1 · Laminar Flow**

This is the only notebook with TODOs. Everything afterwards calls what you
write here, so get it right before moving on.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.1-burgers-2d/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The two residuals

Take the gradient of each component **once** and slice it: columns 0, 1, 2 are
the $x$, $y$ and $t$ derivatives. Then use `d2` for the viscous terms.

Two equations, and they are coupled through the convection terms: $v$ appears
in the $u$ residual and $u$ in the $v$ residual. That coupling is why one
network with two outputs is the natural first choice.

### Your turn

In [ ]:
# TODO 1 --- the two Burgers residuals ---------------------------------------------------------
# Two `...` to replace (one gradient call per component, then slice it):
#   line 1  ->  gu[:,2:3] + u*gu[:,0:1] + v*gu[:,1:2] - nu*(d2(u,xyt,0)+d2(u,xyt,1))    u_t + u u_x + v u_y - nu lap(u)
#   line 2  ->  gv[:,2:3] + u*gv[:,0:1] + v*gv[:,1:2] - nu*(d2(v,xyt,0)+d2(v,xyt,1))    the same for v
def residual_fn(model, xyt, nu):
    out = model(xyt)
    u, v = out[:, 0:1], out[:, 1:2]
    gu, gv = grad(u, xyt), grad(v, xyt)          # columns: x, y, t
    res_u = ...                                   # <- gu[:,2:3] + u*gu[:,0:1] + v*gu[:,1:2] - nu*(d2(u,xyt,0)+d2(u,xyt,1))
    res_v = ...                                   # <- gv[:,2:3] + u*gv[:,0:1] + v*gv[:,1:2] - nu*(d2(v,xyt,0)+d2(v,xyt,1))
    return res_u, res_v
# ------------------------------------------------------------------------------

## 2 · The loss

Two residual terms plus a boundary/initial term taken from the exact solution.
`loss_fn_factory` must return a zero-argument callable, because that is what
the trainer expects: `train_two_stage` calls it repeatedly inside the L-BFGS
line search, and it has to close over the model and the points.

The boundary values are computed **once**, outside the closure. They do not
change between steps, and recomputing them every call would be the most
expensive line in the loop.

### Your turn

In [ ]:
# TODO 2 --- the loss ------------------------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  mse(res_u) + mse(res_v)                                   the physics, both components
#   line 2  ->  mse(out[:,0:1] - ub) + mse(out[:,1:2] - vb)                the boundary values, both components
# Both fields and the residual are O(1) here, so no weight between the terms.
def loss_fn_factory(model, xyt_f, xyt_b, nu):
    xb = to_numpy(xyt_b)
    ub = to_tensor(pb.exact_u(xb[:,0], xb[:,1], xb[:,2], nu).reshape(-1,1))
    vb = to_tensor(pb.exact_v(xb[:,0], xb[:,1], xb[:,2], nu).reshape(-1,1))

    def loss_fn():
        res_u, res_v = residual_fn(model, xyt_f, nu)
        L_pde = ...                               # <- mse(res_u) + mse(res_v)
        out = model(xyt_b)
        L_bc  = ...                               # <- mse(out[:,0:1] - ub) + mse(out[:,1:2] - vb)
        return L_pde + L_bc
    return loss_fn
# ------------------------------------------------------------------------------

## 3 · Check it on the default configuration

In [ ]:
cfg = pb.Config(n_collocation=3000, reynolds=20, adam_epochs=1500, lbfgs_epochs=150)
result = pb.run_study(cfg, residual_fn, loss_fn_factory)

plot_curves(result["history"], title="2-D Burgers — Adam, then L-BFGS")
plt.show()

In [ ]:
pb.plot_fields(result, t=0.5)

## 4 · Save

Notebook 04 collects every run in this set into one report, so each notebook
writes its results to `Ex09.1_outputs`.

In [ ]:
import pickle

os.makedirs(pb.OUTPUT_DIR, exist_ok=True)
path = os.path.join(pb.OUTPUT_DIR, "nb01_baseline.pkl")
with open(path, "wb") as f:
    pickle.dump({k: v for k, v in result.items() if k != "model"}, f)
torch.save(result["model"].state_dict(),
           os.path.join(pb.OUTPUT_DIR, "nb01_baseline.pt"))
print("wrote", path)

## 5 · Before you move on

1. Where is the error largest — in the smooth regions or along the front?
   Explain it using slide 14 on sampling.
2. `u + v = 3/2` for the exact solution, everywhere and for all time. Check
   whether your model obeys it. Nothing in the loss asked it to.
3. The loss has no weight between its terms. Look at the size of the residual
   and of the boundary mismatch at the end of training and say whether one
   term is doing all the work.

Next: **notebook 02**, where the same two functions are driven from sliders.